# Leadership and Management Book Recommendation System

## Notebook 01: API Data Collection

### Purpose

This notebook collects leadership and management book metadata from a book API.

The initial target is approximately **1,000 book records** related to leadership, management, strategy, organizational behavior, communication, entrepreneurship, people management, change management, and related professional development topics.

The API collection process will begin with a small sample. The sample will be inspected before the collection process is scaled.

This approach helps verify:

1. API accessibility
2. Available fields
3. Data quality
4. Missing values
5. Duplicate records
6. ISBN availability
7. Rating coverage
8. Price coverage
9. Description quality
10. Book cover availability
11. Publication information
12. Language information

The raw API response will be transformed into a structured dataset and saved without performing major data cleaning.

Major cleaning and standardization will be completed in a later notebook.

## 1. Data Source

### Primary API

The primary API selected for the initial collection is the **Google Books API**.

The API provides structured metadata for books and book editions.

Depending on data availability, potential variables include:

1. Google Books ID
2. Title
3. Subtitle
4. Authors
5. Publisher
6. Published date
7. Description
8. ISBN 10
9. ISBN 13
10. Page count
11. Categories
12. Average rating
13. Ratings count
14. Language
15. Book cover
16. Sale status
17. List price
18. Retail price
19. Currency
20. Book information URL

Not every book is expected to contain every variable.

Missing API information will remain missing during the raw data collection stage.

## 2. Collection Strategy

A single search term may produce a biased dataset dominated by one type of leadership or management book.

To improve topic coverage, multiple search queries will be used.

Potential search topics include:

### Leadership

1. Leadership
2. Leadership Development
3. Executive Leadership
4. Team Leadership
5. Transformational Leadership
6. Servant Leadership
7. Coaching
8. Emotional Intelligence

### Management

1. Management
2. People Management
3. Strategic Management
4. Change Management
5. Operations Management
6. Project Management
7. Performance Management
8. Human Resource Management

### Business and Organizational Topics

1. Organizational Behavior
2. Business Strategy
3. Decision Making
4. Communication
5. Team Management
6. Innovation
7. Entrepreneurship
8. Organizational Culture

Records collected from the different queries will later be combined and deduplicated.

ISBN 13 will be used as an important identifier where available, together with other identifiers such as the Google Books volume ID.

## 3. Raw Data Principle

This notebook is responsible for **data acquisition**, not full data cleaning.

The API data will therefore be preserved as closely as practical to the information returned by the source.

Major transformations such as:

1. Duplicate removal
2. Missing value treatment
3. Category standardization
4. Author standardization
5. Date correction
6. Price conversion
7. Popularity score construction
8. Leadership and management classification

will be performed in later notebooks.

The expected raw output file is:

`data/raw/api_books_raw.csv`

## 4. Import Libraries

### Purpose

Import the Python libraries required for API requests, data handling, timing, and basic inspection.

### Libraries

`requests` will be used to communicate with the API.

`pandas` will be used to organize the collected records into a DataFrame.

`time` will allow controlled pauses between requests when necessary.

`pathlib` will provide reproducible project file paths.

In [1]:
import requests
import pandas as pd
import time
import pathlib

### Explanation

The imported libraries have the following purposes:

`requests` sends HTTP requests to the book API.

`pandas` converts collected book records into structured tabular data.

`time` can introduce pauses between API requests when required.

`pathlib` helps manage project folders and file paths without relying on computer specific absolute paths.

### Expected Result

The cell should run without an error.

No output is expected from the import cell.

## 5. Confirm Project Path

### Purpose

Before collecting data, confirm that the notebook can correctly identify the project directory.

This is important because the project should remain reproducible when another analyst clones the repository.

In [2]:
project_path = pathlib.Path.cwd().parent

print("Project path:")
print(project_path)

Project path:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System


In [3]:
raw_data_path = project_path / "data" / "raw"

print("Raw data folder:")
print(raw_data_path)

print("\nFolder exists:")
print(raw_data_path.exists())

Raw data folder:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/raw

Folder exists:
True


## 6. API Connection Test

### Purpose

Before collecting the complete dataset, a small request will be sent to the Google Books API.

The purpose of this test is to confirm that:

1. The API endpoint is accessible.
2. The request returns a successful HTTP status code.
3. Leadership and management books are returned.
4. The JSON response contains the expected book metadata.
5. The available fields can support the requirements of the project.

Only a small sample will be requested at this stage.

## 7. Alternative API Evaluation

Following the Google Books API quota limitation identified during the initial connection test, the Open Library API will be evaluated as an alternative primary data source.

Open Library was included among the recommended API sources for the Book Recommendation System project.

A small sample will first be retrieved to evaluate:

1. Leadership and management relevance
2. Title coverage
3. Author information
4. ISBN availability
5. Publication information
6. Language information
7. Subject information
8. Ratings information
9. Edition information
10. Data completeness

The API will only be scaled to the full collection target after the sample has been inspected.

In [4]:
open_library_url = "https://openlibrary.org/search.json"

params = {
    "q": "leadership management",
    "limit": 5
}

response_ol = requests.get(
    open_library_url,
    params=params,
    timeout=30
)

print("Status code:", response_ol.status_code)

Status code: 200


## 8. Inspect the Open Library API Response

### Purpose

The Open Library API returned HTTP status code `200`, confirming that the API is accessible.

Before extracting variables, the JSON response will be inspected to understand its structure and determine which fields are available for the Leadership and Management Book Recommendation System.

No data transformation will be performed at this stage.

## 9. Check Search Results

### Purpose

The number of books matching the search query will be inspected.

The API may identify many matching records, while only five records were requested for the initial sample.

This distinction is important because the total number of matches does not represent the number of records actually collected.

In [5]:
open_library_data = response_ol.json()

print(type(open_library_data))
print(open_library_data.keys())

<class 'dict'>
dict_keys(['numFound', 'start', 'numFoundExact', 'num_found', 'documentation_url', 'q', 'offset', 'docs'])


In [6]:
print("Total matching records:", open_library_data.get("numFound"))
print("Records returned:", len(open_library_data.get("docs", [])))

Total matching records: 15515
Records returned: 5


## 10. Inspect the First Book Record

### Purpose

The first book returned by the Open Library API will be examined in its raw form.

The objective is to identify the available metadata fields before designing the extraction process.

At this stage, the data will only be inspected. No cleaning or transformation will be performed.

In [7]:
first_ol_book = open_library_data["docs"][0]

print(type(first_ol_book))
print(first_ol_book.keys())

<class 'dict'>
dict_keys(['author_key', 'author_name', 'cover_edition_key', 'cover_i', 'ebook_access', 'edition_count', 'first_publish_year', 'has_fulltext', 'ia', 'ia_collection', 'key', 'language', 'lending_edition_s', 'lending_identifier_s', 'public_scan_b', 'title'])


In [8]:
first_ol_book

{'author_key': ['OL3543802A'],
 'author_name': ['Alfie Morgan'],
 'cover_edition_key': 'OL11091643M',
 'cover_i': 5095489,
 'ebook_access': 'borrowable',
 'edition_count': 1,
 'first_publish_year': 2001,
 'has_fulltext': True,
 'ia': ['strategicleaders0000morg'],
 'ia_collection': ['inlibrary', 'internetarchivebooks', 'printdisabled'],
 'key': '/works/OL9539838W',
 'language': ['eng'],
 'lending_edition_s': 'OL11091643M',
 'lending_identifier_s': 'strategicleaders0000morg',
 'public_scan_b': False,
 'title': 'Strategic Leadership'}

## 11. Inspect Important Book Variables

### Purpose

Selected variables will be examined to determine whether Open Library provides sufficient metadata for the project.

Particular attention will be given to identifiers, authorship, publication information, subjects, languages, ratings, and edition information.

In [9]:
print("Title:", first_ol_book.get("title"))
print("Authors:", first_ol_book.get("author_name"))
print("First publication year:", first_ol_book.get("first_publish_year"))
print("Publishers:", first_ol_book.get("publisher"))
print("Languages:", first_ol_book.get("language"))
print("Subjects:", first_ol_book.get("subject"))
print("ISBN:", first_ol_book.get("isbn"))
print("Ratings average:", first_ol_book.get("ratings_average"))
print("Ratings count:", first_ol_book.get("ratings_count"))
print("Edition count:", first_ol_book.get("edition_count"))

Title: Strategic Leadership
Authors: ['Alfie Morgan']
First publication year: 2001
Publishers: None
Languages: ['eng']
Subjects: None
ISBN: None
Ratings average: None
Ratings count: None
Edition count: 1


## 12. Sample Relevance and Data Quality Inspection

### Purpose

The five sample records will be reviewed to assess their relevance to the Leadership and Management Book Recommendation System.

The inspection will also provide an initial assessment of metadata completeness.

The following characteristics will be reviewed:

1. Title
2. Author
3. First publication year
4. Subjects
5. Average rating
6. Ratings count
7. Edition count
8. Languages

Only the first ten subjects will be displayed to keep the inspection output readable.

In [10]:
for number, book in enumerate(open_library_data.get("docs", []), start=1):

    subjects = book.get("subject", [])

    print(f"BOOK {number}")
    print("Title:", book.get("title"))
    print("Authors:", book.get("author_name"))
    print("First publication year:", book.get("first_publish_year"))
    print("Subjects:", subjects[:10])
    print("Ratings average:", book.get("ratings_average"))
    print("Ratings count:", book.get("ratings_count"))
    print("Edition count:", book.get("edition_count"))
    print("Languages:", book.get("language"))
    print("-" * 70)

BOOK 1
Title: Strategic Leadership
Authors: ['Alfie Morgan']
First publication year: 2001
Subjects: []
Ratings average: None
Ratings count: None
Edition count: 1
Languages: ['eng']
----------------------------------------------------------------------
BOOK 2
Title: Leadership, Management and Data Science
Authors: ['Jerry Ramonyai']
First publication year: 2022
Subjects: []
Ratings average: None
Ratings count: None
Edition count: 9
Languages: ['eng']
----------------------------------------------------------------------
BOOK 3
Title: Leadership, Management and Data Science
Authors: ['Jerry Ramonyai']
First publication year: 2022
Subjects: []
Ratings average: None
Ratings count: None
Edition count: 12
Languages: ['eng']
----------------------------------------------------------------------
BOOK 4
Title: Leadership, management and the five essentials for success
Authors: ['Rick Joyner']
First publication year: 1990
Subjects: []
Ratings average: None
Ratings count: None
Edition count: 2
La

## 13. Initial Sample Evaluation

The initial Open Library API test successfully returned book records related to the search query.

The sample demonstrated good availability of basic bibliographic information, including:

1. Book title
2. Author
3. First publication year
4. Language
5. Edition count

However, several important data quality limitations were identified.

### Missing Metadata

All five sample records had missing values for:

1. Subjects
2. Average rating
3. Ratings count

This suggests that these variables may not be consistently available through the default Open Library search results.

### Duplicate and Edition Issues

The sample also contained two records with the same title and author:

`Leadership, Management and Data Science`

The records had different edition counts, indicating that duplicate works or different bibliographic records may appear within the search results.

Deduplication will therefore be necessary during the data integration and cleaning stages.

### Search Relevance

The broad query `leadership management` produced relevant books but also returned specialized titles from particular professional fields.

The full data collection should therefore not depend on a single search query.

A multi query collection strategy will be evaluated to improve topic diversity and relevance.

### Initial Decision

Open Library is suitable as an API data source for bibliographic information, but the current sample does not provide sufficient evidence that it can independently supply all variables required for the final recommendation system.

Additional API fields, endpoints, and external data sources will therefore be evaluated before full scale collection.

## 14. Search Query Evaluation

### Purpose

The initial combined query produced relevant results but also demonstrated limited metadata completeness and potential topic bias.

Several individual leadership and management queries will therefore be tested.

The purpose is to determine whether targeted queries produce more relevant and diverse books than a single broad query.

Only five records will be retrieved for each query during this evaluation.

In [11]:
test_queries = [
    "leadership",
    "management",
    "strategic management",
    "organizational behavior",
    "change management",
    "emotional intelligence"
]

In [12]:
query_results = {}

for query in test_queries:

    params = {
        "q": query,
        "limit": 5
    }

    response = requests.get(
        open_library_url,
        params=params,
        timeout=30
    )

    print(f"{query}: {response.status_code}")

    if response.status_code == 200:
        query_results[query] = response.json()

    time.sleep(1)

leadership: 200
management: 200
strategic management: 200
organizational behavior: 200
change management: 200
emotional intelligence: 200


## 15. Compare Query Results

### Purpose

The titles returned by each test query will be compared to determine which search terms provide the most relevant books for the project.

This evaluation will help define the final API collection strategy before large scale data acquisition.

In [13]:
for query, result in query_results.items():

    print(f"\nQUERY: {query.upper()}")
    print("=" * 70)

    for number, book in enumerate(result.get("docs", []), start=1):
        print(
            number,
            "|",
            book.get("title"),
            "|",
            book.get("author_name")
        )


QUERY: LEADERSHIP
1 | Principle-Centered Leadership | ['Stephen R. Covey']
2 | Leadership in Organizations | ['Gary A. Yukl']
3 | Kepemimpinan = | ['Karjadi M.']
4 | Spiritual leadership | ['J. Oswald Sanders']
5 | Leadership | ['Peter Guy Northouse']

QUERY: MANAGEMENT
1 | Marketing management | ['Philip Kotler']
2 | Management information systems | ['Kenneth C. Laudon', 'Jane P. Laudon', 'Jane Price Laudon', 'Jane Laudon']
3 | Marketing Management | ['Philip Kotler', 'Kevin Lane Keller']
4 | Human Resource management | ['Gary Dessler']
5 | Operations management | ['Jay H. Heizer', 'Jay Heizer', 'Barry Render']

QUERY: STRATEGIC MANAGEMENT
1 | Strategic management | ['Fred R. David']
2 | Strategic management | ['Gregory G. Dess', 'G. T. Lumpkin', 'Alan B. Eisner', 'Gerry McNamara', 'Seung-Hyun Lee', 'Marilyn L. Taylor']
3 | Strategic management and business policy | ['Thomas L. Wheelen', 'J. David Hunger', 'Tom Wheelen']
4 | Strategic management | ['Arthur A. Thompson', 'Alonzo J. St

## 16. Query Evaluation Findings

The targeted query evaluation produced substantially more relevant results than the initial combined query `leadership management`.

### Leadership

The leadership query returned established leadership literature, including works associated with Stephen R. Covey, Gary Yukl, Peter Northouse, and J. Oswald Sanders.

The results also demonstrated that language and geographic diversity may appear within the search results.

### Management

The management query returned books covering several major management disciplines, including:

1. Marketing Management
2. Management Information Systems
3. Human Resource Management
4. Operations Management

This demonstrates that the broad concept of management contains several distinct subfields that should be represented separately during data collection.

### Strategic Management

The strategic management query returned highly relevant academic and professional books. However, several books shared similar or identical titles.

This reinforces the need for ISBN, work identifiers, edition information, author information, and publication information during deduplication.

### Organizational Behavior

The organizational behavior query produced highly relevant books from established authors in the field.

The results indicate that organizational behavior should be retained as a distinct collection topic.

### Change Management

The change management query produced relevant results but also showed duplicate or closely related titles.

### Emotional Intelligence

The emotional intelligence query produced highly relevant books, including multiple records associated with Daniel Goleman.

The results also demonstrated potential author name inconsistencies that will require standardization during data cleaning.

### Overall Finding

Targeted topic queries provide better control over the relevance and diversity of the API dataset than a single broad search query.

The final API collection will therefore use a portfolio of leadership and management topics rather than one general search term.

## 17. Final API Query Portfolio

### Purpose

A balanced query portfolio will be used to collect books across multiple leadership and management domains.

The objective is not to force exactly 1,000 unique books. Instead, approximately 1,000 raw API records will be collected before cleaning and deduplication.

The collection will target approximately 50 records from each of 20 topic queries.

This provides representation across leadership, management, organizational, strategic, and professional development topics.

In [14]:
collection_queries = [
    "leadership",
    "leadership development",
    "executive leadership",
    "team leadership",
    "transformational leadership",
    "servant leadership",
    "management",
    "people management",
    "strategic management",
    "change management",
    "operations management",
    "project management",
    "performance management",
    "human resource management",
    "organizational behavior",
    "business strategy",
    "decision making",
    "communication",
    "emotional intelligence",
    "innovation management"
]

print("Number of collection queries:", len(collection_queries))

Number of collection queries: 20


In [15]:
target_per_query = 50
target_raw_records = len(collection_queries) * target_per_query

print("Target records per query:", target_per_query)
print("Target raw API records:", target_raw_records)

Target records per query: 50
Target raw API records: 1000


## 18. Full Query Collection Test

### Purpose

Before executing the complete API collection, one topic will be tested using the intended collection size of 50 records.

The leadership query will be used for this test.

The objectives are to confirm:

1. The API can return 50 records successfully.
2. The returned records contain usable metadata.
3. The response can be converted into a DataFrame.
4. Important variables can be extracted consistently.
5. The collection strategy does not introduce unexpected technical problems.

The complete 20 query collection will only be executed after this test is validated.

In [16]:
test_params = {
    "q": "leadership",
    "limit": 50
}

test_response = requests.get(
    open_library_url,
    params=test_params,
    timeout=30
)

print("Status code:", test_response.status_code)

Status code: 200


In [17]:
leadership_test_data = test_response.json()

print("Total matching records:", leadership_test_data.get("numFound"))
print("Records returned:", len(leadership_test_data.get("docs", [])))

Total matching records: 76298
Records returned: 50


## 19. Define Raw API Fields

### Purpose

The Open Library JSON response contains many fields, some of which are nested or stored as lists.

A consistent extraction structure will therefore be created before the complete API collection begins.

The raw API dataset will retain useful bibliographic, descriptive, popularity, language, and identifier information where available.

No missing values will be artificially filled during this stage.

### Target Raw Variables

The initial Open Library dataset will include:

1. `openlibrary_key`
2. `title`
3. `authors`
4. `author_keys`
5. `first_publish_year`
6. `publish_dates`
7. `publishers`
8. `isbn_10`
9. `isbn_13`
10. `all_isbns`
11. `languages`
12. `subjects`
13. `edition_count`
14. `ratings_average`
15. `ratings_count`
16. `ratings_count_1`
17. `ratings_count_2`
18. `ratings_count_3`
19. `ratings_count_4`
20. `ratings_count_5`
21. `want_to_read_count`
22. `currently_reading_count`
23. `already_read_count`
24. `cover_id`
25. `ebook_access`
26. `has_fulltext`
27. `public_scan`
28. `collection_query`
29. `source`
30. `source_url`

The exact availability of these variables will be evaluated using the 50 record test dataset.

## 20. Create the Book Extraction Function

### Purpose

A reusable function will convert each Open Library book record into a consistent dictionary.

The function will:

1. Extract the required variables.
2. Separate ISBN 10 and ISBN 13 identifiers.
3. Preserve list based information such as authors, subjects, publishers, and languages.
4. Retain reading and rating indicators where available.
5. Record the query responsible for retrieving the book.
6. Record the original data source.
7. Create a direct Open Library source URL when a work identifier is available.

The function will not perform major cleaning or deduplication.

In [18]:
def extract_open_library_book(book, collection_query):

    # Extract ISBN identifiers
    all_isbns = book.get("isbn", [])

    isbn_10 = [
        isbn for isbn in all_isbns
        if len(str(isbn).replace("-", "")) == 10
    ]

    isbn_13 = [
        isbn for isbn in all_isbns
        if len(str(isbn).replace("-", "")) == 13
    ]

    # Open Library work identifier
    openlibrary_key = book.get("key")

    if openlibrary_key:
        source_url = f"https://openlibrary.org{openlibrary_key}"
    else:
        source_url = None

    return {
        "openlibrary_key": openlibrary_key,
        "title": book.get("title"),
        "authors": book.get("author_name"),
        "author_keys": book.get("author_key"),
        "first_publish_year": book.get("first_publish_year"),
        "publish_dates": book.get("publish_date"),
        "publishers": book.get("publisher"),
        "isbn_10": isbn_10,
        "isbn_13": isbn_13,
        "all_isbns": all_isbns,
        "languages": book.get("language"),
        "subjects": book.get("subject"),
        "edition_count": book.get("edition_count"),
        "ratings_average": book.get("ratings_average"),
        "ratings_count": book.get("ratings_count"),
        "ratings_count_1": book.get("ratings_count_1"),
        "ratings_count_2": book.get("ratings_count_2"),
        "ratings_count_3": book.get("ratings_count_3"),
        "ratings_count_4": book.get("ratings_count_4"),
        "ratings_count_5": book.get("ratings_count_5"),
        "want_to_read_count": book.get("want_to_read_count"),
        "currently_reading_count": book.get("currently_reading_count"),
        "already_read_count": book.get("already_read_count"),
        "cover_id": book.get("cover_i"),
        "ebook_access": book.get("ebook_access"),
        "has_fulltext": book.get("has_fulltext"),
        "public_scan": book.get("public_scan_b"),
        "collection_query": collection_query,
        "source": "Open Library",
        "source_url": source_url
    }

## 21. Test the Extraction Function

### Purpose

The extraction function will first be tested on a single record.

This allows the transformed structure to be inspected before applying the function to all 50 test records.

In [19]:
test_book = leadership_test_data["docs"][0]

extracted_test_book = extract_open_library_book(
    test_book,
    "leadership"
)

extracted_test_book

{'openlibrary_key': '/works/OL2630041W',
 'title': 'Principle-Centered Leadership',
 'authors': ['Stephen R. Covey'],
 'author_keys': ['OL383159A'],
 'first_publish_year': 1989,
 'publish_dates': None,
 'publishers': None,
 'isbn_10': [],
 'isbn_13': [],
 'all_isbns': [],
 'languages': ['eng'],
 'subjects': None,
 'edition_count': 21,
 'ratings_average': None,
 'ratings_count': None,
 'ratings_count_1': None,
 'ratings_count_2': None,
 'ratings_count_3': None,
 'ratings_count_4': None,
 'ratings_count_5': None,
 'want_to_read_count': None,
 'currently_reading_count': None,
 'already_read_count': None,
 'cover_id': 10858615,
 'ebook_access': 'borrowable',
 'has_fulltext': True,
 'public_scan': False,
 'collection_query': 'leadership',
 'source': 'Open Library',
 'source_url': 'https://openlibrary.org/works/OL2630041W'}

## 22. Create the Test DataFrame

### Purpose

The extraction function will now be applied to all 50 leadership records.

The resulting dictionaries will be converted into a pandas DataFrame.

This provides the first structured representation of the API data and allows data quality to be evaluated before full scale collection.

In [20]:
leadership_records = []

for book in leadership_test_data.get("docs", []):

    record = extract_open_library_book(
        book,
        "leadership"
    )

    leadership_records.append(record)

leadership_test_df = pd.DataFrame(leadership_records)

print("Shape:", leadership_test_df.shape)

leadership_test_df.head()

Shape: (50, 30)


,openlibrary_key,title,authors,author_keys,first_publish_year,publish_dates,publishers,isbn_10,isbn_13,all_isbns,...,want_to_read_count,currently_reading_count,already_read_count,cover_id,ebook_access,has_fulltext,public_scan,collection_query,source,source_url
0,/works/OL2630041W,Principle-Centered Leadership,[Stephen R. Covey],[OL383159A],1989,None,None,[],[],[],...,None,None,None,10858615,borrowable,True,False,leadership,Open Library,https://openlibrary.org/works/OL2630041W
1,/works/OL2731767W,Leadership in Organizations,[Gary A. Yukl],[OL400156A],1981,None,None,[],[],[],...,None,None,None,87719,borrowable,True,False,leadership,Open Library,https://openlibrary.org/works/OL2731767W
2,/works/OL302757W,Kepemimpinan =,[Karjadi M.],[OL1268A],1977,None,None,[],[],[],...,None,None,None,14420782,no_ebook,False,False,leadership,Open Library,https://openlibrary.org/works/OL302757W
3,/works/OL450702W,Spiritual leadership,[J. Oswald Sanders],[OL25389A],1967,None,None,[],[],[],...,None,None,None,570509,printdisabled,True,False,leadership,Open Library,https://openlibrary.org/works/OL450702W
4,/works/OL94176W,Leadership,[Peter Guy Northouse],[OL32040A],1997,None,None,[],[],[],...,None,None,None,3859675,printdisabled,True,False,leadership,Open Library,https://openlibrary.org/works/OL94176W


In [21]:
leadership_test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   openlibrary_key          50 non-null     str   
 1   title                    50 non-null     str   
 2   authors                  50 non-null     object
 3   author_keys              50 non-null     object
 4   first_publish_year       50 non-null     int64 
 5   publish_dates            0 non-null      object
 6   publishers               0 non-null      object
 7   isbn_10                  50 non-null     object
 8   isbn_13                  50 non-null     object
 9   all_isbns                50 non-null     object
 10  languages                50 non-null     object
 11  subjects                 0 non-null      object
 12  edition_count            50 non-null     int64 
 13  ratings_average          0 non-null      object
 14  ratings_count            0 non-null      object
 15  ra

In [22]:
leadership_test_df.columns.tolist()

['openlibrary_key',
 'title',
 'authors',
 'author_keys',
 'first_publish_year',
 'publish_dates',
 'publishers',
 'isbn_10',
 'isbn_13',
 'all_isbns',
 'languages',
 'subjects',
 'edition_count',
 'ratings_average',
 'ratings_count',
 'ratings_count_1',
 'ratings_count_2',
 'ratings_count_3',
 'ratings_count_4',
 'ratings_count_5',
 'want_to_read_count',
 'currently_reading_count',
 'already_read_count',
 'cover_id',
 'ebook_access',
 'has_fulltext',
 'public_scan',
 'collection_query',
 'source',
 'source_url']

## 23. Initial Missing Data Assessment

### Purpose

The completeness of each extracted variable will be evaluated before the API collection is scaled.

The analysis will calculate:

1. Number of missing records
2. Percentage of missing records
3. Number of populated records

This assessment will help determine which Open Library variables can serve as core analytical features and which variables will require enrichment from other sources.

No missing values will be treated or imputed at this stage.

In [23]:
missing_summary = pd.DataFrame({
    "missing_count": leadership_test_df.isna().sum(),
    "missing_pct": (
        leadership_test_df.isna().mean() * 100
    ).round(2),
    "available_count": leadership_test_df.notna().sum()
})

missing_summary.sort_values(
    "missing_pct",
    ascending=False
)

,missing_count,missing_pct,available_count
ratings_count_1,50,100.0,0
already_read_count,50,100.0,0
ratings_count_3,50,100.0,0
ratings_count_2,50,100.0,0
want_to_read_count,50,100.0,0
ratings_count,50,100.0,0
ratings_average,50,100.0,0
currently_reading_count,50,100.0,0
subjects,50,100.0,0
ratings_count_4,50,100.0,0


## 24. Actual Field Availability Assessment

### Purpose

The initial pandas missing value assessment does not identify empty Python lists as missing values.

Several Open Library fields, including ISBNs and other list based variables, may therefore appear complete even when no information is actually available.

A second availability assessment will treat the following as unavailable:

1. `None`
2. `NaN`
3. Empty lists
4. Empty strings

This provides a more accurate assessment of API metadata coverage.

In [24]:
def has_value(value):

    if value is None:
        return False

    if isinstance(value, (list, tuple, set, dict)):
        return len(value) > 0

    if isinstance(value, str):
        return value.strip() != ""

    return pd.notna(value)

In [25]:
availability_summary = []

for column in leadership_test_df.columns:

    available = leadership_test_df[column].apply(has_value).sum()
    total = len(leadership_test_df)

    availability_summary.append({
        "field": column,
        "available_count": available,
        "missing_count": total - available,
        "availability_pct": round((available / total) * 100, 2)
    })

availability_df = pd.DataFrame(availability_summary)

availability_df.sort_values(
    "availability_pct",
    ascending=True
)

,field,available_count,missing_count,availability_pct
14,ratings_count,0,50,0.0
18,ratings_count_4,0,50,0.0
17,ratings_count_3,0,50,0.0
16,ratings_count_2,0,50,0.0
15,ratings_count_1,0,50,0.0
21,currently_reading_count,0,50,0.0
13,ratings_average,0,50,0.0
22,already_read_count,0,50,0.0
11,subjects,0,50,0.0
19,ratings_count_5,0,50,0.0


## 25. Expanded Field Request Test

### Purpose

The initial Open Library search response provided useful basic bibliographic information but did not populate several variables required for the project.

An expanded API request will therefore explicitly request additional metadata fields.

The expanded request will be tested on only five leadership books before modifying the full extraction process.

In [26]:
requested_fields = [
    "key",
    "title",
    "author_name",
    "author_key",
    "first_publish_year",
    "publish_date",
    "publisher",
    "isbn",
    "language",
    "subject",
    "edition_count",
    "ratings_average",
    "ratings_count",
    "ratings_count_1",
    "ratings_count_2",
    "ratings_count_3",
    "ratings_count_4",
    "ratings_count_5",
    "want_to_read_count",
    "currently_reading_count",
    "already_read_count",
    "cover_i",
    "ebook_access",
    "has_fulltext",
    "public_scan_b"
]

expanded_params = {
    "q": "leadership",
    "limit": 5,
    "fields": ",".join(requested_fields)
}

expanded_response = requests.get(
    open_library_url,
    params=expanded_params,
    timeout=30
)

print("Status code:", expanded_response.status_code)

Status code: 200


In [27]:
expanded_data = expanded_response.json()

print("Records returned:", len(expanded_data.get("docs", [])))

Records returned: 5


In [28]:
expanded_book = expanded_data["docs"][0]

print("Available keys:")
print(expanded_book.keys())

Available keys:
dict_keys(['author_key', 'author_name', 'cover_i', 'ebook_access', 'edition_count', 'first_publish_year', 'has_fulltext', 'isbn', 'key', 'language', 'public_scan_b', 'publish_date', 'publisher', 'title', 'subject', 'ratings_average', 'ratings_count', 'ratings_count_1', 'ratings_count_2', 'ratings_count_3', 'ratings_count_4', 'ratings_count_5', 'want_to_read_count', 'currently_reading_count', 'already_read_count'])


In [29]:
print("Title:", expanded_book.get("title"))
print("Authors:", expanded_book.get("author_name"))
print("Publisher:", expanded_book.get("publisher"))
print("Publish dates:", expanded_book.get("publish_date"))
print("ISBN:", expanded_book.get("isbn"))
print("Languages:", expanded_book.get("language"))
print("Subjects:", expanded_book.get("subject"))
print("Average rating:", expanded_book.get("ratings_average"))
print("Ratings count:", expanded_book.get("ratings_count"))
print("Want to read:", expanded_book.get("want_to_read_count"))
print("Currently reading:", expanded_book.get("currently_reading_count"))
print("Already read:", expanded_book.get("already_read_count"))

Title: Principle-Centered Leadership
Authors: ['Stephen R. Covey']
Publisher: ['Franklin Covey on Brilliance Audio', 'Free Press', 'Covey', 'Coach Series', 'Pocket Books', 'Simon & Schuster Audio', 'Simon Schuster Trade', 'Cedar Fort', 'Simon & Schuster Inc.', 'Covey Leadership Center', 'Simon & Schuster (Trade Division)', 'Simon & Schuster', 'Summit Books']
Publish dates: ['January 1, 1992', 'October 1, 2001', 'March 1, 1992', '1991', 'September 1991', 'January 1, 2002', 'April 1989', 'January 4, 1999', 'March 1, 2005', 'October 1, 1992', '2002', '2003', 'March 7, 2005', 'January 1, 2000', 'Apr 01, 2012', '1992', 'October 26, 1992']
ISBN: ['9780671792800', '9780671711351', '9781555170486', '9780028639123', '9781883219062', '9780671317034', '9780671755454', '002863912X', '9780743468602', '9780671711160', '9780743501552', '0671011138', '9781929494613', '9780916095314', '068485841X', '9781883219246', '0743501551', '1883219248', '9780684858418', '1929494610', '0916095312', '1596590084', '

## 27. Expanded API Metadata Finding

The expanded Open Library API request successfully returned substantially richer metadata than the default search response.

For the test record, the expanded response included:

1. Multiple publishers
2. Multiple publication dates
3. ISBN 10 and ISBN 13 identifiers
4. Language information
5. Subject classifications
6. Average rating
7. Ratings count
8. Reading interest indicators
9. Edition information
10. Cover information

This demonstrates that the initial missing metadata was primarily caused by the limited default field selection rather than complete absence of the information from Open Library.

### Important Analytical Consideration

Metadata availability does not necessarily indicate statistical reliability.

For example, a book may have a high average rating but only a very small number of ratings. Average rating must therefore be interpreted together with ratings count.

The project will preserve the original rating variables and evaluate appropriate popularity and weighted rating measures during feature engineering.

### Decision

The expanded field request will be used for the API collection.

Before collecting approximately 1,000 raw records, the expanded request will be tested on 50 records to evaluate metadata coverage across a larger sample.

In [30]:
def extract_open_library_book(book, collection_query):

    all_isbns = book.get("isbn") or []

    isbn_10 = [
        isbn for isbn in all_isbns
        if len(str(isbn).replace("-", "")) == 10
    ]

    isbn_13 = [
        isbn for isbn in all_isbns
        if len(str(isbn).replace("-", "")) == 13
    ]

    openlibrary_key = book.get("key")

    source_url = (
        f"https://openlibrary.org{openlibrary_key}"
        if openlibrary_key
        else None
    )

    cover_id = book.get("cover_i")

    cover_url = (
        f"https://covers.openlibrary.org/b/id/{cover_id}-L.jpg"
        if cover_id
        else None
    )

    return {
        "openlibrary_key": openlibrary_key,
        "title": book.get("title"),
        "authors": book.get("author_name"),
        "author_keys": book.get("author_key"),
        "first_publish_year": book.get("first_publish_year"),
        "publish_dates": book.get("publish_date"),
        "publishers": book.get("publisher"),
        "isbn_10": isbn_10,
        "isbn_13": isbn_13,
        "all_isbns": all_isbns,
        "languages": book.get("language"),
        "subjects": book.get("subject"),
        "edition_count": book.get("edition_count"),
        "ratings_average": book.get("ratings_average"),
        "ratings_count": book.get("ratings_count"),
        "ratings_count_1": book.get("ratings_count_1"),
        "ratings_count_2": book.get("ratings_count_2"),
        "ratings_count_3": book.get("ratings_count_3"),
        "ratings_count_4": book.get("ratings_count_4"),
        "ratings_count_5": book.get("ratings_count_5"),
        "want_to_read_count": book.get("want_to_read_count"),
        "currently_reading_count": book.get("currently_reading_count"),
        "already_read_count": book.get("already_read_count"),
        "cover_id": cover_id,
        "cover_url": cover_url,
        "ebook_access": book.get("ebook_access"),
        "has_fulltext": book.get("has_fulltext"),
        "public_scan": book.get("public_scan_b"),
        "collection_query": collection_query,
        "source": "Open Library API",
        "source_url": source_url
    }

## 29. Expanded 50 Record Collection Test

### Purpose

The expanded field configuration will now be tested on 50 leadership books.

This test will determine the actual metadata coverage before the complete multi query API collection is executed.

The resulting dataset will be evaluated for:

1. Record count
2. Field availability
3. Identifier coverage
4. Subject coverage
5. Rating coverage
6. Reading interest coverage
7. Publisher coverage
8. Publication information
9. Cover availability
10. Potential duplicate records

In [31]:
expanded_50_params = {
    "q": "leadership",
    "limit": 50,
    "fields": ",".join(requested_fields)
}

expanded_50_response = requests.get(
    open_library_url,
    params=expanded_50_params,
    timeout=30
)

print("Status code:", expanded_50_response.status_code)

Status code: 200


In [32]:
expanded_50_data = expanded_50_response.json()

print(
    "Records returned:",
    len(expanded_50_data.get("docs", []))
)

Records returned: 50


In [33]:
expanded_records = []

for book in expanded_50_data.get("docs", []):

    record = extract_open_library_book(
        book,
        "leadership"
    )

    expanded_records.append(record)

expanded_50_df = pd.DataFrame(expanded_records)

print("Shape:", expanded_50_df.shape)

expanded_50_df.head()

Shape: (50, 31)


,openlibrary_key,title,authors,author_keys,first_publish_year,publish_dates,publishers,isbn_10,isbn_13,all_isbns,...,currently_reading_count,already_read_count,cover_id,cover_url,ebook_access,has_fulltext,public_scan,collection_query,source,source_url
0,/works/OL2630041W,Principle-Centered Leadership,[Stephen R. Covey],[OL383159A],1989,"[January 1, 1992, October 1, 2001, March 1, 19...","[Franklin Covey on Brilliance Audio, Free Pres...","[002863912X, 0671011138, 068485841X, 074350155...","[9780671792800, 9780671711351, 9781555170486, ...","[9780671792800, 9780671711351, 9781555170486, ...",...,15,6,10858615,https://covers.openlibrary.org/b/id/10858615-L...,borrowable,True,False,leadership,Open Library API,https://openlibrary.org/works/OL2630041W
1,/works/OL2731767W,Leadership in Organizations,[Gary A. Yukl],[OL400156A],1981,"[2010, 2007, 1994, 2013, 1989, 2017-01-01, Jun...","[Prentice Hall, Prentice-Hall International, P...","[0138157146, 0132771861, 0536867038, 013530874...","[9780132424318, 9780130323125, 9780131494848, ...","[0138157146, 9780132424318, 0132771861, 978013...",...,12,0,87719,https://covers.openlibrary.org/b/id/87719-L.jpg,borrowable,True,False,leadership,Open Library API,https://openlibrary.org/works/OL2731767W
2,/works/OL302757W,Kepemimpinan =,[Karjadi M.],[OL1268A],1977,[1977],[Politeia],[],[],[],...,2,18,14420782,https://covers.openlibrary.org/b/id/14420782-L...,no_ebook,False,False,leadership,Open Library API,https://openlibrary.org/works/OL302757W
3,/works/OL450702W,Spiritual leadership,[J. Oswald Sanders],[OL25389A],1967,"[January 1974, 1980, January 2006, 1994, 1999,...","[STL Books, Marshall Pickering, Moody Press, H...","[055100651X, 1596441801, 0767394496, 090384346...","[9780551006515, 9780802482228, 9781596441804, ...","[055100651X, 9780551006515, 1596441801, 978080...",...,7,2,570509,https://covers.openlibrary.org/b/id/570509-L.jpg,printdisabled,True,False,leadership,Open Library API,https://openlibrary.org/works/OL450702W
4,/works/OL94176W,Leadership,[Peter Guy Northouse],[OL32040A],1997,"[2007, 1997, 2010, 2001]","[SAGE Publications, Sage Publications]","[0803957688, 0761919260, 141294161X, 076191925...","[9780803957688, 9781412941617, 9780803957695, ...","[0803957688, 9780803957688, 0761919260, 141294...",...,4,3,3859675,https://covers.openlibrary.org/b/id/3859675-L.jpg,printdisabled,True,False,leadership,Open Library API,https://openlibrary.org/works/OL94176W


## 31. Expanded Metadata Availability Assessment

### Purpose

The expanded 50 record dataset will be evaluated for actual metadata availability.

Both conventional missing values and empty collections will be treated as unavailable.

This analysis will determine which variables are sufficiently populated for the final project and which variables may require enrichment from web scraping or additional sources.

In [34]:
expanded_availability = []

for column in expanded_50_df.columns:

    available = expanded_50_df[column].apply(has_value).sum()
    total = len(expanded_50_df)

    expanded_availability.append({
        "field": column,
        "available_count": available,
        "missing_count": total - available,
        "availability_pct": round(
            (available / total) * 100,
            2
        )
    })

expanded_availability_df = pd.DataFrame(
    expanded_availability
)

expanded_availability_df.sort_values(
    "availability_pct",
    ascending=False
)

,field,available_count,missing_count,availability_pct
0,openlibrary_key,50,0,100.0
1,title,50,0,100.0
29,source,50,0,100.0
28,collection_query,50,0,100.0
27,public_scan,50,0,100.0
26,has_fulltext,50,0,100.0
25,ebook_access,50,0,100.0
24,cover_url,50,0,100.0
23,cover_id,50,0,100.0
22,already_read_count,50,0,100.0


## 32. Preliminary Duplicate Assessment

### Purpose

Open Library work identifiers will be examined for duplicate records within the 50 book test dataset.

This is only a preliminary duplicate assessment.

Full deduplication will be performed during the Data Cleaning and Data Integration stages using multiple identifiers and bibliographic characteristics.

In [35]:
print(
    "Total records:",
    len(expanded_50_df)
)

print(
    "Unique Open Library works:",
    expanded_50_df["openlibrary_key"].nunique()
)

print(
    "Duplicate work IDs:",
    expanded_50_df["openlibrary_key"].duplicated().sum()
)

Total records: 50
Unique Open Library works: 50
Duplicate work IDs: 0


## 33. Open Library Work Endpoint Test

### Purpose

The Open Library Search API provides strong bibliographic metadata but does not include a book description in the current search response.

Descriptions and synopses are particularly important for the recommendation system because they can support:

1. Text cleaning
2. TF IDF analysis
3. Keyword extraction
4. Semantic embeddings
5. Cosine similarity
6. Natural language recommendations

The Open Library Work endpoint will therefore be tested to determine whether descriptions can be retrieved using the work identifiers collected from the Search API.

Only one book will initially be tested.

In [36]:
test_work_key = expanded_50_df.loc[0, "openlibrary_key"]

test_work_url = f"https://openlibrary.org{test_work_key}.json"

print("Work key:", test_work_key)
print("Work API URL:", test_work_url)

Work key: /works/OL2630041W
Work API URL: https://openlibrary.org/works/OL2630041W.json


In [37]:
work_response = requests.get(
    test_work_url,
    timeout=30
)

print("Status code:", work_response.status_code)

Status code: 200


## 34. Inspect the Work Record

### Purpose

The complete Work API response will be inspected to identify additional metadata that is not available through the Search API.

Particular attention will be given to:

1. Description
2. Subjects
3. Subject places
4. Subject people
5. Subject times
6. Covers
7. Creation and revision information

In [38]:
work_data = work_response.json()

print(type(work_data))
print(work_data.keys())

<class 'dict'>
dict_keys(['first_publish_date', 'key', 'title', 'authors', 'type', 'covers', 'first_sentence', 'lc_classifications', 'dewey_number', 'subjects', 'excerpts', 'description', 'latest_revision', 'revision', 'created', 'last_modified'])


In [39]:
work_data

{'first_publish_date': 'March 1, 2005',
 'key': '/works/OL2630041W',
 'title': 'Principle-Centered Leadership',
 'authors': [{'author': {'key': '/authors/OL383159A'},
   'type': {'key': '/type/author_role'}}],
 'type': {'key': '/type/work'},
 'covers': [10858615, 876830, 408834, 408292, 8338000, 479673, 10716342],
 'first_sentence': {'type': '/type/text',
  'value': 'I HAVE LONG ADVOCATED a natural, gradual, day-by-day, step-by-step, sequential approach to personal development.'},
 'lc_classifications': ['BF637.S8 C67 1991'],
 'dewey_number': ['158/.4', '303.34'],
 'subjects': ['Leadership',
  'Psychological aspects of Success',
  'Success',
  'Psychological aspects',
  'Commerce',
  'Success in business',
  'Aptitude pour la direction',
  'Achievement',
  'Success--'],
 'excerpts': [{'excerpt': 'I HAVE LONG ADVOCATED a natural, gradual, day-by-day, step-by-step, sequential approach to personal development.'}],
 'description': {'type': '/type/text',
  'value': 'How do we as individuals

In [40]:
description = work_data.get("description")

if isinstance(description, dict):
    description = description.get("value")

print("Description:")
print(description)

Description:
How do we as individuals and organizations survive and thrive amid tremendous change? Why are efforts to improve falling so short in real results despite the millions of dollars in time, capital, and human effort being spent on them? How do we unleash the creativity, talent, and energy within ourselves and others in the midst of pressure? Is it realistic to believe that balance among personal, family, and professional life is possible? The answer to these and other dilemmas is Principle-Centered Leadership, a long-term, inside-out approach to developing people and organizations. The key to dealing with the challenges that face us today is the recognition of a principle-centered core within both ourselves and our organizations. Dr. Covey offers insights and guidelines that can help you apply these principles both at work and at home -- leading not just to a new understanding of how to increase quality and productivity, but also to a new appreciation of the importance of bui

In [41]:
print("Subjects:", work_data.get("subjects"))
print("Subject places:", work_data.get("subject_places"))
print("Subject people:", work_data.get("subject_people"))
print("Subject times:", work_data.get("subject_times"))
print("Covers:", work_data.get("covers"))

Subjects: ['Leadership', 'Psychological aspects of Success', 'Success', 'Psychological aspects', 'Commerce', 'Success in business', 'Aptitude pour la direction', 'Achievement', 'Success--']
Subject places: None
Subject people: None
Subject times: None
Covers: [10858615, 876830, 408834, 408292, 8338000, 479673, 10716342]


## 36. Description Availability Test

### Purpose

A sample of ten Open Library works will be queried to estimate description availability.

The objective is to determine whether the Work endpoint provides sufficient textual information to justify enrichment of the complete API dataset.

A one second delay will be used between requests to avoid unnecessary request frequency.

In [42]:
description_test_results = []

for _, row in expanded_50_df.head(10).iterrows():

    work_key = row["openlibrary_key"]
    work_url = f"https://openlibrary.org{work_key}.json"

    response = requests.get(
        work_url,
        timeout=30
    )

    description = None

    if response.status_code == 200:

        work = response.json()

        description = work.get("description")

        if isinstance(description, dict):
            description = description.get("value")

    description_test_results.append({
        "openlibrary_key": work_key,
        "title": row["title"],
        "status_code": response.status_code,
        "description": description,
        "has_description": bool(description)
    })

    time.sleep(1)

In [43]:
description_test_df = pd.DataFrame(
    description_test_results
)

description_test_df[
    [
        "title",
        "status_code",
        "has_description"
    ]
]

,title,status_code,has_description
0,Principle-Centered Leadership,200,True
1,Leadership in Organizations,200,False
2,Kepemimpinan =,200,False
3,Spiritual leadership,200,False
4,Leadership,200,False
5,The 21 Irrefutable Laws of Leadership,200,False
6,Leadership,200,True
7,Leadership and performance beyond expectations,200,False
8,A Higher Loyalty,200,True
9,Leadership and Self Deception,200,True


In [44]:
description_available = (
    description_test_df["has_description"].sum()
)

description_total = len(description_test_df)

description_pct = (
    description_available / description_total * 100
)

print("Books tested:", description_total)
print("Descriptions available:", description_available)
print(
    "Description availability:",
    round(description_pct, 2),
    "%"
)

Books tested: 10
Descriptions available: 4
Description availability: 40.0 %


## 37. Work Endpoint Evaluation

The Open Library Work endpoint was evaluated as a potential source of additional textual metadata.

The endpoint successfully returned detailed information for the test book, including:

1. Description
2. Subjects
3. First sentence
4. Excerpts
5. Library of Congress classifications
6. Dewey Decimal classifications
7. Cover identifiers
8. Creation and revision information

### Description Availability

Ten books from the leadership sample were tested.

Results:

- Books tested: 10
- Descriptions available: 4
- Description availability: 40%

The Work endpoint therefore provides useful descriptive information, but description coverage is incomplete.

### Decision

The Work endpoint will be retained as an enrichment source because descriptions are valuable for NLP and recommendation modeling.

However, descriptions will be treated as optional metadata rather than a mandatory inclusion criterion.

Books without descriptions will not be removed solely because this field is unavailable.

Additional descriptions may later be obtained through the web scraping dataset or other enrichment sources.

## 38. Final API Collection Strategy

Based on the API testing stage, Open Library will be used as the primary API source for the project.

### Search API

The Search API will provide the main bibliographic dataset.

Twenty targeted leadership and management queries will be used, with approximately 50 records requested per query.

Target raw collection:

20 queries × 50 records = approximately 1,000 raw API records

Duplicate books across queries are expected and will be preserved in the initial raw collection.

### Work API

The Work endpoint will subsequently be used to enrich unique Open Library works with additional textual metadata where available.

Potential enrichment fields include:

- Description
- First sentence
- Subjects
- Excerpts
- Library of Congress classifications
- Dewey Decimal classifications

### Data Integrity Principle

Raw API responses will not be artificially completed or imputed.

Missing information will remain missing until the Data Cleaning, Data Integration, or Feature Engineering stages.

The original collection query and source information will be retained for provenance and reproducibility.

## 39. Collect the Full API Dataset

### Purpose

The validated Search API configuration will now be applied to the complete query portfolio.

Approximately 50 records will be requested for each of the 20 leadership and management topics.

The collection process will:

1. Request expanded metadata fields.
2. Preserve the originating search query.
3. Apply the validated extraction function.
4. Record request status and record counts.
5. Use controlled request pacing.
6. Preserve duplicate records at this stage.

No deduplication or major cleaning will be performed during collection.

In [45]:
api_records = []
collection_log = []

for query in collection_queries:

    params = {
        "q": query,
        "limit": target_per_query,
        "fields": ",".join(requested_fields)
    }

    try:
        response = requests.get(
            open_library_url,
            params=params,
            timeout=30
        )

        status_code = response.status_code

        if status_code == 200:

            data = response.json()
            books = data.get("docs", [])

            for book in books:

                record = extract_open_library_book(
                    book,
                    query
                )

                api_records.append(record)

            collection_log.append({
                "query": query,
                "status_code": status_code,
                "records_collected": len(books)
            })

            print(
                f"{query}: "
                f"{len(books)} records collected"
            )

        else:

            collection_log.append({
                "query": query,
                "status_code": status_code,
                "records_collected": 0
            })

            print(
                f"{query}: "
                f"request failed with status {status_code}"
            )

    except requests.RequestException as error:

        collection_log.append({
            "query": query,
            "status_code": None,
            "records_collected": 0
        })

        print(
            f"{query}: request error - {error}"
        )

    time.sleep(1)

leadership: 50 records collected
leadership development: 50 records collected
executive leadership: 50 records collected
team leadership: 50 records collected
transformational leadership: 50 records collected
servant leadership: 50 records collected
management: 50 records collected
people management: 50 records collected
strategic management: 50 records collected
change management: 50 records collected
operations management: 50 records collected
project management: 50 records collected
performance management: 50 records collected
human resource management: 50 records collected
organizational behavior: 50 records collected
business strategy: 50 records collected
decision making: 50 records collected
communication: 50 records collected
emotional intelligence: 50 records collected
innovation management: 50 records collected


## 40. Create the Raw API DataFrame

The collected API records will be converted into a pandas DataFrame.

The DataFrame represents the raw structured Search API dataset before deduplication, cleaning, or enrichment.

In [46]:
api_raw_df = pd.DataFrame(api_records)

print("Raw API dataset shape:", api_raw_df.shape)

api_raw_df.head()

Raw API dataset shape: (1000, 31)


,openlibrary_key,title,authors,author_keys,first_publish_year,publish_dates,publishers,isbn_10,isbn_13,all_isbns,...,currently_reading_count,already_read_count,cover_id,cover_url,ebook_access,has_fulltext,public_scan,collection_query,source,source_url
0,/works/OL2630041W,Principle-Centered Leadership,[Stephen R. Covey],[OL383159A],1989.0,"[January 1, 1992, October 1, 2001, March 1, 19...","[Franklin Covey on Brilliance Audio, Free Pres...","[002863912X, 0671011138, 068485841X, 074350155...","[9780671792800, 9780671711351, 9781555170486, ...","[9780671792800, 9780671711351, 9781555170486, ...",...,15.0,6.0,10858615.0,https://covers.openlibrary.org/b/id/10858615-L...,borrowable,True,False,leadership,Open Library API,https://openlibrary.org/works/OL2630041W
1,/works/OL2731767W,Leadership in Organizations,[Gary A. Yukl],[OL400156A],1981.0,"[2010, 2007, 1994, 2013, 1989, 2017-01-01, Jun...","[Prentice Hall, Prentice-Hall International, P...","[0138157146, 0132771861, 0536867038, 013530874...","[9780132424318, 9780130323125, 9780131494848, ...","[0138157146, 9780132424318, 0132771861, 978013...",...,12.0,0.0,87719.0,https://covers.openlibrary.org/b/id/87719-L.jpg,borrowable,True,False,leadership,Open Library API,https://openlibrary.org/works/OL2731767W
2,/works/OL302757W,Kepemimpinan =,[Karjadi M.],[OL1268A],1977.0,[1977],[Politeia],[],[],[],...,2.0,18.0,14420782.0,https://covers.openlibrary.org/b/id/14420782-L...,no_ebook,False,False,leadership,Open Library API,https://openlibrary.org/works/OL302757W
3,/works/OL450702W,Spiritual leadership,[J. Oswald Sanders],[OL25389A],1967.0,"[January 1974, 1980, January 2006, 1994, 1999,...","[STL Books, Marshall Pickering, Moody Press, H...","[055100651X, 1596441801, 0767394496, 090384346...","[9780551006515, 9780802482228, 9781596441804, ...","[055100651X, 9780551006515, 1596441801, 978080...",...,7.0,2.0,570509.0,https://covers.openlibrary.org/b/id/570509-L.jpg,printdisabled,True,False,leadership,Open Library API,https://openlibrary.org/works/OL450702W
4,/works/OL94176W,Leadership,[Peter Guy Northouse],[OL32040A],1997.0,"[2007, 1997, 2010, 2001]","[SAGE Publications, Sage Publications]","[0803957688, 0761919260, 141294161X, 076191925...","[9780803957688, 9781412941617, 9780803957695, ...","[0803957688, 9780803957688, 0761919260, 141294...",...,4.0,3.0,3859675.0,https://covers.openlibrary.org/b/id/3859675-L.jpg,printdisabled,True,False,leadership,Open Library API,https://openlibrary.org/works/OL94176W


In [47]:
collection_log_df = pd.DataFrame(collection_log)

collection_log_df

,query,status_code,records_collected
0,leadership,200,50
1,leadership development,200,50
2,executive leadership,200,50
3,team leadership,200,50
4,transformational leadership,200,50
5,servant leadership,200,50
6,management,200,50
7,people management,200,50
8,strategic management,200,50
9,change management,200,50


In [48]:
print(
    "Successful queries:",
    (collection_log_df["status_code"] == 200).sum()
)

print(
    "Total records collected:",
    collection_log_df["records_collected"].sum()
)

Successful queries: 20
Total records collected: 1000


## 42. Preliminary Cross Query Duplicate Assessment

Because the same book may be relevant to multiple search topics, duplicate Open Library work identifiers are expected across the complete API collection.

Duplicates will not yet be removed.

This assessment measures the degree of overlap between the collection queries before formal data cleaning.

In [49]:
total_raw_records = len(api_raw_df)

unique_work_ids = api_raw_df[
    "openlibrary_key"
].nunique()

duplicate_work_records = api_raw_df[
    "openlibrary_key"
].duplicated().sum()

print("Total raw records:", total_raw_records)
print("Unique Open Library works:", unique_work_ids)
print("Duplicate work records:", duplicate_work_records)

Total raw records: 1000
Unique Open Library works: 950
Duplicate work records: 50


In [50]:
duplicate_pct = (
    duplicate_work_records
    / total_raw_records
    * 100
)

print(
    "Duplicate work percentage:",
    round(duplicate_pct, 2),
    "%"
)

Duplicate work percentage: 5.0 %


In [51]:
query_distribution = (
    api_raw_df["collection_query"]
    .value_counts()
    .rename_axis("collection_query")
    .reset_index(name="records")
)

query_distribution

,collection_query,records
0,leadership,50
1,leadership development,50
2,executive leadership,50
3,team leadership,50
4,transformational leadership,50
5,servant leadership,50
6,management,50
7,people management,50
8,strategic management,50
9,change management,50


## 44. Save the Raw Search API Dataset

The structured Search API dataset will be saved to the project's raw data directory.

This file represents the original structured API collection and will not be overwritten by later cleaning or feature engineering operations.

List based metadata will be preserved in the raw CSV representation.

A separate enriched dataset will later be created after Work endpoint enrichment.

In [52]:
api_raw_file = (
    raw_data_path
    / "open_library_search_api_raw.csv"
)

api_raw_df.to_csv(
    api_raw_file,
    index=False
)

print("File saved:")
print(api_raw_file)

File saved:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/raw/open_library_search_api_raw.csv


In [53]:
collection_log_file = (
    raw_data_path
    / "open_library_collection_log.csv"
)

collection_log_df.to_csv(
    collection_log_file,
    index=False
)

print("Collection log saved:")
print(collection_log_file)

Collection log saved:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/raw/open_library_collection_log.csv


## 45. API Collection Summary

The primary Open Library Search API collection was successfully completed.

### Collection Results

- Search queries: 20
- Successful queries: 20
- Records requested per query: 50
- Total raw API records: 1,000
- Dataset variables: 31
- Unique Open Library works: 950
- Duplicate work occurrences: 50
- Duplicate work percentage: 5.0%

Each search query successfully returned 50 records, resulting in a balanced raw collection across the selected leadership and management topics.

### Raw Files Created

1. `open_library_search_api_raw.csv`
   - Contains the 1,000 raw Search API records.

2. `open_library_collection_log.csv`
   - Contains the query level API collection results.

### Important Data Principle

The 50 duplicate work occurrences will remain in the raw dataset.

Raw data will not be modified or overwritten.

Deduplication will be performed later during the Data Cleaning stage after collection and enrichment have been completed.

## 46. Prepare Unique Works for Work API Enrichment

### Purpose

The Search API dataset contains 1,000 raw records representing 950 unique Open Library works.

Calling the Work endpoint for every raw record would unnecessarily repeat requests for books retrieved by multiple search queries.

A unique work list will therefore be created using the Open Library work identifier.

This list will be used only for API enrichment and does not modify the original raw Search API dataset.

In [54]:
unique_works_df = (
    api_raw_df[
        ["openlibrary_key", "title"]
    ]
    .drop_duplicates(
        subset="openlibrary_key"
    )
    .reset_index(drop=True)
)

print(
    "Unique works prepared for enrichment:",
    len(unique_works_df)
)

unique_works_df.head()

Unique works prepared for enrichment: 950


,openlibrary_key,title
0,/works/OL2630041W,Principle-Centered Leadership
1,/works/OL2731767W,Leadership in Organizations
2,/works/OL302757W,Kepemimpinan =
3,/works/OL450702W,Spiritual leadership
4,/works/OL94176W,Leadership


## 47. Define Work Metadata Extraction

The Work API enrichment process will collect additional textual and classification metadata where available.

Target enrichment variables include:

- Description
- First sentence
- Work subjects
- Subject places
- Subject people
- Subject times
- Excerpts
- Library of Congress classifications
- Dewey Decimal classifications
- Work cover identifiers

Missing metadata will remain missing.

In [55]:
def extract_text_value(value):

    if isinstance(value, dict):
        return value.get("value")

    return value

In [56]:
def extract_work_metadata(work):

    description = extract_text_value(
        work.get("description")
    )

    first_sentence = extract_text_value(
        work.get("first_sentence")
    )

    excerpts = work.get("excerpts") or []

    excerpt_texts = []

    for excerpt in excerpts:

        if isinstance(excerpt, dict):

            text = extract_text_value(
                excerpt.get("excerpt")
            )

            if text:
                excerpt_texts.append(text)

    return {
        "description": description,
        "first_sentence": first_sentence,
        "work_subjects": work.get("subjects"),
        "subject_places": work.get("subject_places"),
        "subject_people": work.get("subject_people"),
        "subject_times": work.get("subject_times"),
        "excerpts": excerpt_texts,
        "lc_classifications": work.get(
            "lc_classifications"
        ),
        "dewey_number": work.get("dewey_number"),
        "work_covers": work.get("covers")
    }

## 48. Collect Work API Enrichment Data

### Purpose

Each unique Open Library work will be queried once through the Work endpoint.

The enrichment process will:

1. Retrieve additional textual metadata.
2. Preserve the Open Library work identifier.
3. Record HTTP status codes.
4. Continue when an individual request fails.
5. Apply controlled request pacing.
6. Save progress periodically.

Periodic checkpoint files reduce the risk of losing collected enrichment data if the process is interrupted.

In [57]:
unique_works_df = (
    api_raw_df[
        ["openlibrary_key", "title"]
    ]
    .drop_duplicates(
        subset="openlibrary_key"
    )
    .reset_index(drop=True)
)

print(
    "Unique works prepared for enrichment:",
    len(unique_works_df)
)

unique_works_df.head()

Unique works prepared for enrichment: 950


,openlibrary_key,title
0,/works/OL2630041W,Principle-Centered Leadership
1,/works/OL2731767W,Leadership in Organizations
2,/works/OL302757W,Kepemimpinan =
3,/works/OL450702W,Spiritual leadership
4,/works/OL94176W,Leadership


In [58]:
work_enrichment_records = []

checkpoint_file = (
    raw_data_path
    / "open_library_work_enrichment_checkpoint.csv"
)

for index, row in unique_works_df.iterrows():

    work_key = row["openlibrary_key"]
    title = row["title"]

    work_url = (
        f"https://openlibrary.org"
        f"{work_key}.json"
    )

    record = {
        "openlibrary_key": work_key,
        "title": title,
        "work_status_code": None
    }

    try:

        response = requests.get(
            work_url,
            timeout=30
        )

        record["work_status_code"] = (
            response.status_code
        )

        if response.status_code == 200:

            work = response.json()

            metadata = extract_work_metadata(
                work
            )

            record.update(metadata)

    except requests.RequestException as error:

        record["request_error"] = str(error)

    work_enrichment_records.append(record)

    # Progress display
    if (index + 1) % 50 == 0:

        print(
            f"{index + 1} / "
            f"{len(unique_works_df)} "
            f"works processed"
        )

    # Save checkpoint every 100 records
    if (index + 1) % 100 == 0:

        checkpoint_df = pd.DataFrame(
            work_enrichment_records
        )

        checkpoint_df.to_csv(
            checkpoint_file,
            index=False
        )

    time.sleep(1)

50 / 950 works processed
100 / 950 works processed
150 / 950 works processed
200 / 950 works processed
250 / 950 works processed
300 / 950 works processed
350 / 950 works processed
400 / 950 works processed
450 / 950 works processed
500 / 950 works processed
550 / 950 works processed
600 / 950 works processed
650 / 950 works processed
700 / 950 works processed
750 / 950 works processed
800 / 950 works processed
850 / 950 works processed
900 / 950 works processed
950 / 950 works processed


In [59]:
work_enrichment_df = pd.DataFrame(
    work_enrichment_records
)

print(
    "Work enrichment shape:",
    work_enrichment_df.shape
)

work_enrichment_df.head()

Work enrichment shape: (950, 13)


,openlibrary_key,title,work_status_code,description,first_sentence,work_subjects,subject_places,subject_people,subject_times,excerpts,lc_classifications,dewey_number,work_covers
0,/works/OL2630041W,Principle-Centered Leadership,200,How do we as individuals and organizations sur...,"I HAVE LONG ADVOCATED a natural, gradual, day-...","[Leadership, Psychological aspects of Success,...",None,None,None,"[I HAVE LONG ADVOCATED a natural, gradual, day...",[BF637.S8 C67 1991],"[158/.4, 303.34]","[10858615, 876830, 408834, 408292, 8338000, 47..."
1,/works/OL2731767W,Leadership in Organizations,200,NaN,NaN,"[Organisation, Prise de décision, Entscheidung...",None,None,None,[],None,[303.3/4],"[87719, 1116320, 9752736, 11666429, 12221566, ..."
2,/works/OL302757W,Kepemimpinan =,200,NaN,NaN,[Leadership],None,None,None,[],[HM141 .K32],None,None
3,/works/OL450702W,Spiritual leadership,200,NaN,NaN,"[Christian leadership, Leadership, Spiritual d...",None,None,None,[],None,None,[570509]
4,/works/OL94176W,Leadership,200,NaN,NaN,"[Cas, Études de, Leiderschap, Leadership, Case...",None,None,None,[],None,None,[3859675]


In [60]:
work_status_summary = (
    work_enrichment_df[
        "work_status_code"
    ]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

work_status_summary.columns = [
    "status_code",
    "records"
]

work_status_summary

,status_code,records
0,200,950


In [61]:
successful_work_requests = (
    work_enrichment_df[
        "work_status_code"
    ]
    .eq(200)
    .sum()
)

print(
    "Successful Work API requests:",
    successful_work_requests
)

print(
    "Total unique works:",
    len(work_enrichment_df)
)

Successful Work API requests: 950
Total unique works: 950


In [62]:
description_available = (
    work_enrichment_df[
        "description"
    ]
    .apply(has_value)
    .sum()
)

description_total = len(
    work_enrichment_df
)

description_pct = (
    description_available
    / description_total
    * 100
)

print(
    "Descriptions available:",
    description_available
)

print(
    "Total unique works:",
    description_total
)

print(
    "Description availability:",
    round(description_pct, 2),
    "%"
)

Descriptions available: 192
Total unique works: 950
Description availability: 20.21 %


In [63]:
enrichment_fields = [
    "description",
    "first_sentence",
    "work_subjects",
    "subject_places",
    "subject_people",
    "subject_times",
    "excerpts",
    "lc_classifications",
    "dewey_number",
    "work_covers"
]

enrichment_coverage = []

for field in enrichment_fields:

    available = (
        work_enrichment_df[field]
        .apply(has_value)
        .sum()
    )

    total = len(work_enrichment_df)

    enrichment_coverage.append({
        "field": field,
        "available_count": available,
        "missing_count": total - available,
        "availability_pct": round(
            available / total * 100,
            2
        )
    })

enrichment_coverage_df = pd.DataFrame(
    enrichment_coverage
)

enrichment_coverage_df.sort_values(
    "availability_pct",
    ascending=False
)

,field,available_count,missing_count,availability_pct
2,work_subjects,826,124,86.95
9,work_covers,743,207,78.21
0,description,192,758,20.21
8,dewey_number,155,795,16.32
7,lc_classifications,85,865,8.95
3,subject_places,56,894,5.89
1,first_sentence,32,918,3.37
6,excerpts,23,927,2.42
4,subject_people,13,937,1.37
5,subject_times,9,941,0.95


## 52. Save Work API Enrichment Data

The Work API enrichment dataset will be stored separately from the Search API dataset.

Keeping the two raw sources separate preserves provenance and allows the enrichment process to be reproduced independently.

The datasets will not be merged or cleaned within the Data Collection stage.

In [65]:
work_enrichment_file = (
    raw_data_path
    / "open_library_work_enrichment_raw.csv"
)

work_enrichment_df.to_csv(
    work_enrichment_file,
    index=False
)

print("Work enrichment saved:")
print(work_enrichment_file)

Work enrichment saved:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/raw/open_library_work_enrichment_raw.csv


In [66]:
work_enrichment_file = (
    raw_data_path
    / "open_library_work_enrichment_raw.csv"
)

work_enrichment_df.to_csv(
    work_enrichment_file,
    index=False
)

print("Work enrichment saved:")
print(work_enrichment_file)

Work enrichment saved:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/raw/open_library_work_enrichment_raw.csv


## 53. Final API Data Collection Summary

### Search API Collection

The Open Library Search API was used as the primary API source for the Leadership and Management Book Recommendation System.

Twenty targeted leadership and management search queries were executed.

Final Search API results:

- Search queries: 20
- Successful queries: 20
- Records collected per query: 50
- Total raw Search API records: 1,000
- Variables collected: 31
- Unique Open Library works: 950
- Duplicate work occurrences: 50
- Duplicate percentage: 5.0%

Duplicate records were intentionally retained in the raw Search API dataset because raw source data should not be modified during collection.

### Work API Enrichment

The 950 unique Open Library work identifiers were subsequently queried through the Open Library Work API.

Final Work API results:

- Unique works requested: 950
- Successful requests: 950
- Failed requests: 0
- Request success rate: 100%

The Work API provided additional textual and classification metadata.

### Enrichment Coverage

| Field | Available Records | Availability |
|---|---:|---:|
| Work subjects | 826 | 86.95% |
| Work covers | 743 | 78.21% |
| Description | 192 | 20.21% |
| Dewey number | 155 | 16.32% |
| LC classifications | 85 | 8.95% |
| Subject places | 56 | 5.89% |
| First sentence | 32 | 3.37% |
| Excerpts | 23 | 2.42% |
| Subject people | 13 | 1.37% |
| Subject times | 9 | 0.95% |

### Key Finding

The Work API provides strong subject metadata but limited description coverage.

Descriptions are available for 20.21% of unique works and therefore cannot be treated as a mandatory textual feature.

Work subjects, with 86.95% availability, provide a substantially stronger source of thematic information.

During later NLP and feature engineering stages, multiple textual variables may be combined, including title, subjects, and description where available.

Missing descriptions will not be artificially generated or imputed during data collection.

## 54. Data Collection Limitations

The API collection process has several limitations that must be considered during subsequent analysis.

### 1. Search Query Dependence

The dataset was collected using predefined leadership and management search queries.

The resulting dataset therefore reflects the selected query portfolio and should not be interpreted as a complete representation of all leadership and management books ever published.

### 2. Query Overlap

Books may appear under multiple search queries.

This resulted in 50 duplicate work occurrences among the 1,000 raw Search API records.

These duplicates were intentionally preserved during collection and will be addressed during data cleaning.

### 3. Incomplete Ratings

Ratings are not available for every Open Library record.

Average ratings must also be interpreted together with ratings count because a high average based on very few ratings does not provide the same evidence as a similar average based on many ratings.

### 4. Limited Description Coverage

Only 192 of 950 unique works contained descriptions through the Work API, representing 20.21% availability.

Descriptions therefore cannot serve as the sole textual foundation of the recommendation system.

### 5. Metadata Variation

Books may contain multiple:

- ISBNs
- Publishers
- Publication dates
- Languages
- Subjects
- Covers

This occurs because Open Library work records can represent multiple editions of the same underlying work.

Edition-level and work-level information must therefore be interpreted carefully during cleaning and database design.

### 6. Missing Commercial Information

Open Library does not provide all commercial variables required by the project, particularly consistent current book prices.

Commercial information may therefore require collection from the web scraping source.

### 7. Geographic and Translation Information

Author nationality, original publication country, original language, and translation availability cannot be reliably inferred from author names, ISBNs, or language codes alone.

These variables will only be included when supported by verifiable data sources.

### 8. Raw Data Preservation

Missing values were not artificially completed during API collection.

No statistical imputation, deduplication, normalization, or major transformation was performed on the raw API datasets.

## 55. API Output Files

The API collection stage generated the following raw data files:

### `open_library_search_api_raw.csv`

Contains the 1,000 raw records collected through the Open Library Search API.

### `open_library_collection_log.csv`

Contains the query-level collection results, HTTP status codes, and number of records collected.

### `open_library_work_enrichment_raw.csv`

Contains additional Work API metadata for the 950 unique Open Library works.

### `open_library_work_enrichment_checkpoint.csv`

Contains periodic enrichment checkpoints created during the Work API collection process.

These files are stored in:

`data/raw/`

The raw files will remain unchanged during subsequent project stages.

## 56. Next Step

The API data acquisition stage is complete.

The project has successfully collected:

- 1,000 raw Search API records
- 950 unique Open Library works
- Work-level enrichment for all 950 unique works

The next stage of the project is:

# Notebook 02 — Web Scraping

The web scraping stage will target approximately 1,000 additional leadership and management book records.

The scraped dataset should complement the API dataset by providing information that is limited or unavailable through Open Library, particularly where legally and technically accessible.

Potential target variables include:

- Book title
- Author
- Description or synopsis
- Rating
- Rating count
- Price
- Currency
- Category
- Publication information
- Book URL
- Cover image URL
- ISBN where available

The scraped dataset will remain separate from the API datasets until the Data Integration stage.